## Upload many fields

This notebook is an adjusted version of the *create_field* notebook. This adjusted version allows you to take an existing GeoJSON or Shapefile which includes the outlines of many parcels and upload them in one go to a ScoutMaster project.

In [ ]:
# import packages
import sys
import os
from dotenv import load_dotenv
from IPython.display import display, HTML
import json
import matplotlib.pyplot as plt
import folium
import pandas as pd
import geopandas as gpd
from shapely.ops import transform
import pyproj

# Load environment variables from .env file
load_dotenv()

# import the API builder
sys.path.append(os.path.abspath("../../"))
from scoutmasterapi_builder.api import ScoutMasterAPI

In [ ]:
# Scoutmaster API - Credentials
client_id = os.getenv('SM_CLIENT_ID')
client_secret = os.getenv('SM_CLIENT_SECRET')
dev = os.getenv('PROD')

# Scoutmaster API - Init
SM_API = ScoutMasterAPI(dev)
SM_API.authenticate(client_id=client_id, client_secret=client_secret)

In [ ]:
df_projects = SM_API.projects()
df_projects_select = df_projects[df_projects['name'].str.contains('SQAT')]
df_projects_select.head()

In [ ]:
project = df_projects.iloc[140]
project_id = project["id"]

df_users = SM_API.users(project_id)
df_users.head()


In [ ]:
user = df_users.iloc[0]
user_id = user["user.id"]

In [ ]:
print("PROJECT:", project["name"], "ID:", project_id)
print("USER:", user["user.username"], "ID:",  user_id)
# print("FIELD:",     field["name"], "ID:", field["id"])
# print("LAYERTYPE:", layer_type["name"], "ID:", layer_type_id)

In [ ]:
data_path = 'XXX' # Absolute data path
fields_raw = gpd.read_file(data_path)
fields=fields_raw.explode()

In [ ]:
def to_wgs84(geom, source_crs=None):
    """
    Convert a Shapely geometry to WGS:84 (EPSG:4326).
    - If source_crs is given, use it directly.
    - Otherwise, guess: coords within lon/lat range -> assume already WGS84;
      otherwise assume EPSG:3857 (Web Mercator).
    """
    minx, miny, maxx, maxy = geom.bounds

    if source_crs is None:
        if -180 <= minx <= 180 and -180 <= maxx <= 180 and -90 <= miny <= 90 and -90 <= maxy <= 90:
            source_crs = "EPSG:4326"
        else:
            source_crs = "EPSG:3857"

    if pyproj.CRS(source_crs).equals(pyproj.CRS("EPSG:4326")):
        return geom  # already WGS84, no-op

    transformer = pyproj.Transformer.from_crs(source_crs, "EPSG:4326", always_xy=True).transform
    return transform(transformer, geom)


In [ ]:
UPLOAD_TO_SM = True

for index, entry in fields.iterrows():
    # Get the correct column with the field names
    field_name = entry.Naam
    # Convert the geometry to WGS:84
    geometry = to_wgs84(entry.geometry)

    # Create the field dictionary
    field = {
        "user_id": user_id,
        "name": field_name,
        "geometry": geometry.wkt
    }
    print("Ready to upload: ", field)

    if UPLOAD_TO_SM:
        # Upload the field dictionary
        SM_API.fields_create(project_id, field)